# 📊 Sales & Marketing Analytics Project
## IBM SkillsBuild — Data Analytics with AI Internship
**Author:** Prakash Maiti  
**Dataset:** sales.csv (15,000 records × 30 features)  
**Tools:** Python · Pandas · NumPy · Matplotlib · Seaborn · Scikit-learn

---
### 📋 Table of Contents
1. [Setup & Library Imports](#1)
2. [Data Loading & Overview](#2)
3. [Data Cleaning & Preprocessing](#3)
4. [Exploratory Data Analysis (EDA)](#4)
5. [Customer Demographics Analysis](#5)
6. [Revenue & Spending Analysis](#6)
7. [Marketing Channel Effectiveness](#7)
8. [Churn Analysis](#8)
9. [Satisfaction & NPS Analysis](#9)
10. [AI/ML — Churn Prediction Model](#10)
11. [Business Insights & Recommendations](#11)

---
## 1. Setup & Library Imports <a id='1'></a>

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, roc_auc_score, roc_curve)

# Display settings
pd.set_option('display.max_columns', 35)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

print('✅ All libraries imported successfully!')

: 

---
## 2. Data Loading & Overview <a id='2'></a>

In [ ]:
# Load the dataset
df = pd.read_csv('sales.csv')

print(f'📦 Dataset loaded successfully!')
print(f'   Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

In [ ]:
# First look at the data
df.head()

In [ ]:
# Data types and non-null counts
df.info()

In [ ]:
# Statistical summary of numerical columns
df.describe()

In [ ]:
# Check missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

print('🔍 Columns with missing values:')
print(missing_df if not missing_df.empty else '  ✅ No missing values found!')

In [ ]:
# Check for duplicates
dups = df.duplicated().sum()
print(f'🔁 Duplicate rows: {dups}')
print(f'📊 Unique customers: {df["customer_id"].nunique():,}')

---
## 3. Data Cleaning & Preprocessing <a id='3'></a>

In [ ]:
# --- Step 1: Convert date columns ---
df['signup_date'] = pd.to_datetime(df['signup_date'], errors='coerce')
df['last_purchase_date'] = pd.to_datetime(df['last_purchase_date'], errors='coerce')

# --- Step 2: Derive new features ---
df['signup_year'] = df['signup_date'].dt.year
df['signup_month'] = df['signup_date'].dt.month
df['days_since_last_purchase'] = (pd.Timestamp('2025-01-01') - df['last_purchase_date']).dt.days

# --- Step 3: Fill missing gender with 'Unknown' ---
df['gender'] = df['gender'].replace('', np.nan).fillna('Unknown')

# --- Step 4: Fill missing coupon_code with 'None' ---
df['coupon_code'] = df['coupon_code'].replace('', np.nan).fillna('None')

# --- Step 5: Remove duplicate rows (if any) ---
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

# --- Step 6: Age buckets for segmentation ---
df['age_group'] = pd.cut(df['age'],
                          bins=[0, 24, 34, 44, 54, 100],
                          labels=['18-24', '25-34', '35-44', '45-54', '55+'])

print(f'✅ Data cleaned. Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'   New features added: signup_year, signup_month, days_since_last_purchase, age_group')

---
## 4. Exploratory Data Analysis (EDA) <a id='4'></a>

In [ ]:
# Distribution of key numerical features
num_cols = ['age', 'total_spent', 'avg_order_value', 'lifetime_value',
            'avg_session_time', 'total_visits', 'satisfaction_score', 'nps_score']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df[col].dropna(), bins=30, color='steelblue', edgecolor='white', alpha=0.85)
    axes[i].set_title(col.replace('_', ' ').title())
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Count')

plt.suptitle('Distribution of Key Numerical Features', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
corr_cols = ['age', 'total_visits', 'avg_session_time', 'pages_per_session',
             'email_open_rate', 'email_click_rate', 'total_spent', 'avg_order_value',
             'support_tickets', 'satisfaction_score', 'nps_score',
             'marketing_spend_per_user', 'lifetime_value', 'last_3_month_purchase_freq', 'churn']

corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, vmin=-1, vmax=1, linewidths=0.5, annot_kws={'size': 8})
plt.title('Correlation Matrix — Key Numerical Features', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Box plots: Total Spent and LTV by churn status
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

churn_labels = {0: 'Retained', 1: 'Churned'}
df['churn_label'] = df['churn'].map(churn_labels)

sns.boxplot(data=df, x='churn_label', y='total_spent', palette=['#4CAF50', '#F44336'], ax=axes[0])
axes[0].set_title('Total Spent by Churn Status')
axes[0].set_xlabel('Churn Status')
axes[0].set_ylabel('Total Spent (USD)')

sns.boxplot(data=df, x='churn_label', y='lifetime_value', palette=['#4CAF50', '#F44336'], ax=axes[1])
axes[1].set_title('Lifetime Value by Churn Status')
axes[1].set_xlabel('Churn Status')
axes[1].set_ylabel('Lifetime Value (USD)')

plt.suptitle('Spending Patterns: Retained vs Churned Customers', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. Customer Demographics Analysis <a id='5'></a>

In [ ]:
# Gender distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

gender_counts = df['gender'].value_counts()
colors = ['#3b82d4', '#e97d6b', '#a8d5a2']
axes[0].pie(gender_counts, labels=gender_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=140, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[0].set_title('Customer Gender Distribution')

# Age group distribution
age_counts = df['age_group'].value_counts().sort_index()
axes[1].bar(age_counts.index.astype(str), age_counts.values, color='steelblue', edgecolor='white')
axes[1].set_title('Customer Age Group Distribution')
axes[1].set_xlabel('Age Group')
axes[1].set_ylabel('Number of Customers')
for bar, val in zip(axes[1].patches, age_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'{val:,}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Customer Demographics — Gender & Age', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 countries by customer count and revenue
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top_countries = df['country'].value_counts().head(10)
axes[0].barh(top_countries.index[::-1], top_countries.values[::-1], color='#3b82d4')
axes[0].set_title('Top 10 Countries by Customer Count')
axes[0].set_xlabel('Number of Customers')

country_rev = df.groupby('country')['total_spent'].sum().sort_values(ascending=False).head(10)
axes[1].barh(country_rev.index[::-1], country_rev.values[::-1], color='#7c5cd8')
axes[1].set_title('Top 10 Countries by Total Revenue')
axes[1].set_xlabel('Total Revenue (USD)')

plt.suptitle('Country-Level Customer & Revenue Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

: 

In [ ]:
# Device type & Subscription type breakdown
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

device_counts = df['device_type'].value_counts()
colors_d = ['#3b82d4', '#7c5cd8', '#e97d6b']
axes[0].pie(device_counts, labels=device_counts.index, autopct='%1.1f%%',
            colors=colors_d, startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[0].set_title('Device Type Distribution')

sub_counts = df['subscription_type'].value_counts()
axes[1].bar(sub_counts.index, sub_counts.values, color=['#3b82d4', '#e97d6b'], edgecolor='white')
axes[1].set_title('Subscription Type Distribution')
axes[1].set_ylabel('Number of Customers')
for bar, val in zip(axes[1].patches, sub_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'{val:,}', ha='center', va='bottom', fontsize=10)

plt.suptitle('Device & Subscription Preferences', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6. Revenue & Spending Analysis <a id='6'></a>

In [ ]:
# Revenue summary
total_revenue = df['total_spent'].sum()
avg_revenue = df['total_spent'].mean()
median_revenue = df['total_spent'].median()
total_ltv = df['lifetime_value'].sum()

print('=' * 50)
print('       💰 REVENUE SUMMARY')
print('=' * 50)
print(f'  Total Revenue        : ${total_revenue:>12,.2f}')
print(f'  Avg Revenue/Customer : ${avg_revenue:>12,.2f}')
print(f'  Median Revenue       : ${median_revenue:>12,.2f}')
print(f'  Total Lifetime Value : ${total_ltv:>12,.2f}')
print('=' * 50)

In [ ]:
# Revenue by subscription type and premium status
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sub_rev = df.groupby('subscription_type')['total_spent'].mean()
axes[0].bar(sub_rev.index, sub_rev.values, color=['#3b82d4', '#e97d6b'], edgecolor='white', width=0.5)
axes[0].set_title('Avg Total Spent by Subscription Type')
axes[0].set_ylabel('Avg Total Spent (USD)')
for bar, val in zip(axes[0].patches, sub_rev.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 f'${val:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

premium_rev = df.groupby('is_premium_user')[['total_spent', 'lifetime_value']].mean()
premium_rev.index = ['Non-Premium', 'Premium']
x = np.arange(len(premium_rev))
w = 0.35
bars1 = axes[1].bar(x - w/2, premium_rev['total_spent'], w, label='Avg Total Spent', color='#3b82d4', edgecolor='white')
bars2 = axes[1].bar(x + w/2, premium_rev['lifetime_value'], w, label='Avg LTV', color='#7c5cd8', edgecolor='white')
axes[1].set_xticks(x)
axes[1].set_xticklabels(premium_rev.index)
axes[1].set_title('Avg Spent & LTV: Premium vs Non-Premium')
axes[1].set_ylabel('Amount (USD)')
axes[1].legend()

plt.suptitle('Revenue Analysis by Subscription & Premium Status', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Revenue by age group
age_rev = df.groupby('age_group', observed=True)['total_spent'].agg(['mean', 'sum']).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar(age_rev['age_group'].astype(str), age_rev['mean'], color='#3b82d4', edgecolor='white')
axes[0].set_title('Avg Total Spent by Age Group')
axes[0].set_xlabel('Age Group')
axes[0].set_ylabel('Avg Total Spent (USD)')

axes[1].bar(age_rev['age_group'].astype(str), age_rev['sum'], color='#7c5cd8', edgecolor='white')
axes[1].set_title('Total Revenue by Age Group')
axes[1].set_xlabel('Age Group')
axes[1].set_ylabel('Total Revenue (USD)')

plt.suptitle('Revenue Analysis by Age Group', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Payment method analysis
pay_rev = df.groupby('payment_method')['total_spent'].agg(['count', 'mean']).sort_values('mean', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar(pay_rev.index, pay_rev['count'], color='steelblue', edgecolor='white')
axes[0].set_title('Customer Count by Payment Method')
axes[0].set_xlabel('Payment Method')
axes[0].set_ylabel('Number of Customers')
axes[0].tick_params(axis='x', rotation=30)

axes[1].bar(pay_rev.index, pay_rev['mean'], color='#7c5cd8', edgecolor='white')
axes[1].set_title('Avg Spend by Payment Method')
axes[1].set_xlabel('Payment Method')
axes[1].set_ylabel('Avg Total Spent (USD)')
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle('Payment Method Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 7. Marketing Channel Effectiveness <a id='7'></a>

In [ ]:
# Acquisition channel analysis
channel_df = df.groupby('acquisition_channel').agg(
    customer_count=('customer_id', 'count'),
    avg_spent=('total_spent', 'mean'),
    avg_ltv=('lifetime_value', 'mean'),
    avg_marketing_spend=('marketing_spend_per_user', 'mean'),
    churn_rate=('churn', 'mean')
).reset_index()

channel_df['roi'] = (channel_df['avg_ltv'] - channel_df['avg_marketing_spend']) / channel_df['avg_marketing_spend']
channel_df['churn_rate_pct'] = (channel_df['churn_rate'] * 100).round(2)

print('📢 Acquisition Channel Summary:')
print(channel_df[['acquisition_channel', 'customer_count', 'avg_spent', 'avg_ltv',
                   'avg_marketing_spend', 'churn_rate_pct', 'roi']].to_string(index=False))

In [ ]:
# Channel visualisations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ch_sorted_count = channel_df.sort_values('customer_count', ascending=True)
axes[0, 0].barh(ch_sorted_count['acquisition_channel'], ch_sorted_count['customer_count'], color='#3b82d4')
axes[0, 0].set_title('Customers Acquired per Channel')
axes[0, 0].set_xlabel('Number of Customers')

ch_sorted_ltv = channel_df.sort_values('avg_ltv', ascending=True)
axes[0, 1].barh(ch_sorted_ltv['acquisition_channel'], ch_sorted_ltv['avg_ltv'], color='#7c5cd8')
axes[0, 1].set_title('Avg Lifetime Value per Channel')
axes[0, 1].set_xlabel('Avg LTV (USD)')

ch_sorted_churn = channel_df.sort_values('churn_rate_pct', ascending=True)
bar_colors = ['#e97d6b' if v > channel_df['churn_rate_pct'].mean() else '#4CAF50'
              for v in ch_sorted_churn['churn_rate_pct']]
axes[1, 0].barh(ch_sorted_churn['acquisition_channel'], ch_sorted_churn['churn_rate_pct'], color=bar_colors)
axes[1, 0].set_title('Churn Rate per Channel (%)')
axes[1, 0].set_xlabel('Churn Rate (%)')
axes[1, 0].axvline(channel_df['churn_rate_pct'].mean(), color='grey', linestyle='--', alpha=0.7, label='Avg')
axes[1, 0].legend()

ch_sorted_roi = channel_df.sort_values('roi', ascending=True)
roi_colors = ['#4CAF50' if v > 0 else '#e97d6b' for v in ch_sorted_roi['roi']]
axes[1, 1].barh(ch_sorted_roi['acquisition_channel'], ch_sorted_roi['roi'], color=roi_colors)
axes[1, 1].set_title('Marketing ROI per Channel')
axes[1, 1].set_xlabel('ROI (LTV − Mktg Spend) / Mktg Spend')
axes[1, 1].axvline(0, color='black', linewidth=0.8)

plt.suptitle('Marketing Acquisition Channel Effectiveness', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Email marketing performance
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

email_channel = df[df['acquisition_channel'] == 'Email']
axes[0].scatter(df['email_open_rate'], df['total_spent'], alpha=0.3, color='#3b82d4', s=15)
z = np.polyfit(df['email_open_rate'].dropna(), df['total_spent'].dropna(), 1)
p = np.poly1d(z)
x_line = np.linspace(df['email_open_rate'].min(), df['email_open_rate'].max(), 100)
axes[0].plot(x_line, p(x_line), 'r--', linewidth=1.5, label='Trend')
axes[0].set_title('Email Open Rate vs Total Spent')
axes[0].set_xlabel('Email Open Rate')
axes[0].set_ylabel('Total Spent (USD)')
axes[0].legend()

axes[1].scatter(df['email_click_rate'], df['lifetime_value'], alpha=0.3, color='#7c5cd8', s=15)
z2 = np.polyfit(df['email_click_rate'].dropna(), df['lifetime_value'].dropna(), 1)
p2 = np.poly1d(z2)
x_line2 = np.linspace(df['email_click_rate'].min(), df['email_click_rate'].max(), 100)
axes[1].plot(x_line2, p2(x_line2), 'r--', linewidth=1.5, label='Trend')
axes[1].set_title('Email Click Rate vs Lifetime Value')
axes[1].set_xlabel('Email Click Rate')
axes[1].set_ylabel('Lifetime Value (USD)')
axes[1].legend()

plt.suptitle('Email Marketing Performance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 8. Churn Analysis <a id='8'></a>

In [ ]:
# Overall churn rate
churn_rate = df['churn'].mean() * 100
churned = df['churn'].sum()
retained = len(df) - churned

print('=' * 45)
print('       📉 CHURN OVERVIEW')
print('=' * 45)
print(f'  Overall Churn Rate : {churn_rate:.2f}%')
print(f'  Churned Customers  : {churned:,}')
print(f'  Retained Customers : {retained:,}')
print('=' * 45)

In [ ]:
# Churn rate by multiple dimensions
fig, axes = plt.subplots(2, 3, figsize=(17, 10))

def plot_churn_bar(ax, groupby_col, title, observed=False):
    churn_by = df.groupby(groupby_col, observed=observed)['churn'].mean() * 100
    colors = ['#e97d6b' if v > churn_rate else '#4CAF50' for v in churn_by.values]
    churn_by.plot(kind='bar', ax=ax, color=colors, edgecolor='white')
    ax.axhline(churn_rate, color='grey', linestyle='--', linewidth=1, label=f'Avg {churn_rate:.1f}%')
    ax.set_title(title)
    ax.set_ylabel('Churn Rate (%)')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)
    ax.legend(fontsize=8)

plot_churn_bar(axes[0, 0], 'gender', 'Churn Rate by Gender')
plot_churn_bar(axes[0, 1], 'subscription_type', 'Churn Rate by Subscription Type')
plot_churn_bar(axes[0, 2], 'device_type', 'Churn Rate by Device Type')
plot_churn_bar(axes[1, 0], 'acquisition_channel', 'Churn Rate by Acquisition Channel')
plot_churn_bar(axes[1, 1], 'age_group', 'Churn Rate by Age Group', observed=True)
plot_churn_bar(axes[1, 2], 'is_premium_user', 'Churn Rate: Premium vs Non-Premium')

# Fix last label
axes[1, 2].set_xticklabels(['Non-Premium', 'Premium'], rotation=0)

plt.suptitle('Churn Rate Across Customer Segments', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Churn vs Support Tickets & Satisfaction Score
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

churn_support = df.groupby('support_tickets')['churn'].mean() * 100
axes[0].bar(churn_support.index, churn_support.values, color='#e97d6b', edgecolor='white')
axes[0].set_title('Churn Rate by Number of Support Tickets')
axes[0].set_xlabel('Support Tickets')
axes[0].set_ylabel('Churn Rate (%)')

churn_sat = df.groupby('satisfaction_score')['churn'].mean() * 100
axes[1].bar(churn_sat.index, churn_sat.values,
            color=['#e97d6b','#e97d6b','#f5c518','#4CAF50','#4CAF50'], edgecolor='white')
axes[1].set_title('Churn Rate by Satisfaction Score')
axes[1].set_xlabel('Satisfaction Score (1–5)')
axes[1].set_ylabel('Churn Rate (%)')

plt.suptitle('Churn Drivers: Support Issues & Satisfaction', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 9. Satisfaction & NPS Analysis <a id='9'></a>

In [ ]:
# Satisfaction score distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sat_dist = df['satisfaction_score'].value_counts().sort_index()
colors = ['#e97d6b', '#f5a623', '#f5c518', '#a8d5a2', '#4CAF50']
axes[0].bar(sat_dist.index, sat_dist.values, color=colors, edgecolor='white')
axes[0].set_title('Satisfaction Score Distribution')
axes[0].set_xlabel('Satisfaction Score (1–5)')
axes[0].set_ylabel('Number of Customers')
for bar, val in zip(axes[0].patches, sat_dist.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                 f'{val:,}', ha='center', va='bottom', fontsize=9)

# NPS distribution
nps_dist = df['nps_score'].value_counts().sort_index()
nps_colors = ['#e97d6b' if s <= 6 else ('#f5c518' if s <= 8 else '#4CAF50') for s in nps_dist.index]
axes[1].bar(nps_dist.index, nps_dist.values, color=nps_colors, edgecolor='white')
axes[1].set_title('NPS Score Distribution')
axes[1].set_xlabel('NPS Score (0–10)')
axes[1].set_ylabel('Number of Customers')
axes[1].legend(handles=[
    plt.Rectangle((0,0),1,1, color='#e97d6b', label='Detractors (0-6)'),
    plt.Rectangle((0,0),1,1, color='#f5c518', label='Passives (7-8)'),
    plt.Rectangle((0,0),1,1, color='#4CAF50', label='Promoters (9-10)')
], fontsize=8)

plt.suptitle('Customer Satisfaction & NPS Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Net Promoter Score Calculation
promoters = (df['nps_score'] >= 9).sum()
detractors = (df['nps_score'] <= 6).sum()
total = len(df)

nps = ((promoters - detractors) / total) * 100

print('=' * 45)
print('       🌟 NET PROMOTER SCORE (NPS)')
print('=' * 45)
print(f'  Promoters  (9-10) : {promoters:,} ({promoters/total*100:.1f}%)')
print(f'  Passives   (7-8)  : {(df["nps_score"].between(7,8)).sum():,}')
print(f'  Detractors (0-6)  : {detractors:,} ({detractors/total*100:.1f}%)')
print(f'  NPS Score         : {nps:.1f}')
print('=' * 45)
nps_label = 'Excellent' if nps > 50 else ('Good' if nps > 20 else ('Fair' if nps > 0 else 'Needs Improvement'))
print(f'  NPS Category      : {nps_label}')

In [ ]:
# Average satisfaction & NPS by acquisition channel
sat_channel = df.groupby('acquisition_channel')[['satisfaction_score', 'nps_score']].mean().sort_values('satisfaction_score', ascending=False)

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(sat_channel))
w = 0.35
ax.bar(x - w/2, sat_channel['satisfaction_score'], w, label='Avg Satisfaction Score (1–5)', color='#3b82d4', edgecolor='white')
ax2 = ax.twinx()
ax2.bar(x + w/2, sat_channel['nps_score'], w, label='Avg NPS Score (0–10)', color='#7c5cd8', edgecolor='white', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(sat_channel.index, rotation=30, ha='right')
ax.set_ylabel('Avg Satisfaction Score')
ax2.set_ylabel('Avg NPS Score')
ax.set_title('Satisfaction & NPS by Acquisition Channel', fontsize=13, fontweight='bold')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()

---
## 10. AI/ML — Churn Prediction Model <a id='10'></a>

We will build and compare two machine learning models to predict customer churn:
1. **Logistic Regression** — A simple, interpretable baseline model
2. **Random Forest Classifier** — A powerful ensemble model


In [ ]:
# --- Feature Engineering for ML ---

ml_features = [
    'age', 'total_visits', 'avg_session_time', 'pages_per_session',
    'email_open_rate', 'email_click_rate', 'total_spent', 'avg_order_value',
    'discount_used', 'support_tickets', 'refund_requested',
    'delivery_delay_days', 'satisfaction_score', 'nps_score',
    'marketing_spend_per_user', 'lifetime_value', 'last_3_month_purchase_freq',
    'is_premium_user', 'days_since_last_purchase',
    'gender', 'subscription_type', 'device_type', 'acquisition_channel', 'payment_method'
]

target = 'churn'

ml_df = df[ml_features + [target]].copy()

# Encode categorical columns
cat_cols = ['gender', 'subscription_type', 'device_type', 'acquisition_channel', 'payment_method']
le = LabelEncoder()
for col in cat_cols:
    ml_df[col] = le.fit_transform(ml_df[col].astype(str))

# Drop rows with any NaN
ml_df.dropna(inplace=True)

X = ml_df[ml_features]
y = ml_df[target]

# Train/test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale features
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'✅ ML dataset prepared.')
print(f'   Training samples : {len(X_train):,}')
print(f'   Testing samples  : {len(X_test):,}')
print(f'   Features used    : {len(ml_features)}')
print(f'   Churn rate (train): {y_train.mean()*100:.2f}%')

In [ ]:
# --- Model 1: Logistic Regression ---
lr = LogisticRegression(max_iter=500, random_state=42)
lr.fit(X_train_sc, y_train)
y_pred_lr = lr.predict(X_test_sc)
y_prob_lr = lr.predict_proba(X_test_sc)[:, 1]

lr_acc = accuracy_score(y_test, y_pred_lr)
lr_auc = roc_auc_score(y_test, y_prob_lr)

print('🔹 Logistic Regression Results')
print(f'   Accuracy : {lr_acc*100:.2f}%')
print(f'   ROC-AUC  : {lr_auc:.4f}')
print()
print(classification_report(y_test, y_pred_lr, target_names=['Retained', 'Churned']))

In [ ]:
# --- Model 2: Random Forest Classifier ---
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)   # RF does not need scaling
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

rf_acc = accuracy_score(y_test, y_pred_rf)
rf_auc = roc_auc_score(y_test, y_prob_rf)

print('🔸 Random Forest Classifier Results')
print(f'   Accuracy : {rf_acc*100:.2f}%')
print(f'   ROC-AUC  : {rf_auc:.4f}')
print()
print(classification_report(y_test, y_pred_rf, target_names=['Retained', 'Churned']))

In [ ]:
# --- Model Evaluation Plots ---
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Confusion Matrices
for idx, (model_name, y_pred) in enumerate([('Logistic Regression', y_pred_lr), ('Random Forest', y_pred_rf)]):
    cm = confusion_matrix(y_test, y_pred)
    ax = axes[idx]
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Retained', 'Churned'],
                yticklabels=['Retained', 'Churned'])
    ax.set_title(f'Confusion Matrix\n{model_name}')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

# ROC Curves
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)

axes[2].plot(fpr_lr, tpr_lr, label=f'Logistic Reg (AUC={lr_auc:.3f})', color='#3b82d4', lw=2)
axes[2].plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC={rf_auc:.3f})', color='#7c5cd8', lw=2)
axes[2].plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
axes[2].set_title('ROC Curves — Model Comparison')
axes[2].set_xlabel('False Positive Rate')
axes[2].set_ylabel('True Positive Rate')
axes[2].legend(fontsize=9)

plt.suptitle('Churn Prediction Model Evaluation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance (Random Forest)
feat_imp = pd.Series(rf.feature_importances_, index=ml_features).sort_values(ascending=False).head(15)

plt.figure(figsize=(11, 6))
colors = ['#3b82d4' if i < 5 else '#7c5cd8' if i < 10 else '#a8d5a2' for i in range(len(feat_imp))]
plt.barh(feat_imp.index[::-1], feat_imp.values[::-1], color=colors[::-1], edgecolor='white')
plt.title('Top 15 Feature Importances — Random Forest Churn Predictor', fontsize=13, fontweight='bold')
plt.xlabel('Feature Importance Score')
plt.tight_layout()
plt.show()

print('\n🔑 Top 5 Churn Predictors:')
for rank, (feat, imp) in enumerate(feat_imp.head(5).items(), 1):
    print(f'   {rank}. {feat.replace("_", " ").title()}: {imp:.4f}')

In [ ]:
# Model Comparison Summary
print('=' * 50)
print('       🤖 MODEL COMPARISON SUMMARY')
print('=' * 50)
print(f'  Model                  Accuracy   ROC-AUC')
print(f'  ─────────────────────────────────────────')
print(f'  Logistic Regression  : {lr_acc*100:>6.2f}%    {lr_auc:.4f}')
print(f'  Random Forest        : {rf_acc*100:>6.2f}%    {rf_auc:.4f}')
print('=' * 50)
winner = 'Random Forest' if rf_auc > lr_auc else 'Logistic Regression'
print(f'  ✅ Best Model: {winner}')

---
## 11. Business Insights & Recommendations <a id='11'></a>

In [ ]:
# Final Summary Dashboard
fig, axes = plt.subplots(2, 3, figsize=(17, 10))

# 1. Revenue by Channel
ch_rev = df.groupby('acquisition_channel')['total_spent'].sum().sort_values()
axes[0,0].barh(ch_rev.index, ch_rev.values, color='#3b82d4', edgecolor='white')
axes[0,0].set_title('Total Revenue by Acquisition Channel')
axes[0,0].set_xlabel('Total Revenue (USD)')

# 2. Churn rate by premium status
prem_churn = df.groupby('is_premium_user')['churn'].mean() * 100
axes[0,1].bar(['Non-Premium', 'Premium'], prem_churn.values,
              color=['#e97d6b','#4CAF50'], edgecolor='white', width=0.5)
for bar, val in zip(axes[0,1].patches, prem_churn.values):
    axes[0,1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                   f'{val:.1f}%', ha='center', fontweight='bold')
axes[0,1].set_title('Churn Rate: Premium vs Non-Premium')
axes[0,1].set_ylabel('Churn Rate (%)')

# 3. Average LTV by subscription
sub_ltv = df.groupby('subscription_type')['lifetime_value'].mean()
axes[0,2].bar(sub_ltv.index, sub_ltv.values, color=['#7c5cd8','#3b82d4'], edgecolor='white', width=0.5)
for bar, val in zip(axes[0,2].patches, sub_ltv.values):
    axes[0,2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                   f'${val:,.0f}', ha='center', fontweight='bold')
axes[0,2].set_title('Avg Lifetime Value by Subscription')
axes[0,2].set_ylabel('Avg LTV (USD)')

# 4. Satisfaction vs Churn
sat_churn = df.groupby('satisfaction_score')['churn'].mean() * 100
sat_colors = ['#e97d6b','#e97d6b','#f5c518','#4CAF50','#4CAF50']
axes[1,0].bar(sat_churn.index, sat_churn.values, color=sat_colors, edgecolor='white')
axes[1,0].set_title('Churn Rate by Satisfaction Score')
axes[1,0].set_xlabel('Satisfaction Score')
axes[1,0].set_ylabel('Churn Rate (%)')

# 5. Top 5 feature importances
top5 = feat_imp.head(5)
axes[1,1].barh(top5.index[::-1], top5.values[::-1], color='#7c5cd8', edgecolor='white')
axes[1,1].set_title('Top 5 Churn Predictors (ML)')
axes[1,1].set_xlabel('Feature Importance')

# 6. Discount usage impact
disc_churn = df.groupby('discount_used')['churn'].mean() * 100
axes[1,2].bar(['No Discount', 'Discount Used'], disc_churn.values,
              color=['#3b82d4','#e97d6b'], edgecolor='white', width=0.5)
for bar, val in zip(axes[1,2].patches, disc_churn.values):
    axes[1,2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                   f'{val:.1f}%', ha='center', fontweight='bold')
axes[1,2].set_title('Churn Rate: Discount vs No Discount')
axes[1,2].set_ylabel('Churn Rate (%)')

plt.suptitle('Executive Summary Dashboard — Sales & Marketing Analytics', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 💡 Key Business Insights & Recommendations

### 📌 Insight 1: Customer Churn is Satisfaction-Driven
- Customers with satisfaction score **1 or 2** churn at significantly higher rates
- **Recommendation:** Implement a proactive customer support system targeting low-satisfaction customers. Trigger satisfaction surveys after every purchase and escalate scores ≤ 2 to retention teams.

### 📌 Insight 2: Premium Membership Reduces Churn
- Premium users have a **significantly lower churn rate** than non-premium users
- **Recommendation:** Run targeted upgrade campaigns for high-LTV non-premium users. Offer 1-month free Premium trial to at-risk customers.

### 📌 Insight 3: Annual Subscribers are More Valuable
- Annual subscription customers show higher average LTV than monthly subscribers
- **Recommendation:** Offer discounts (10–15%) on annual plans. Promote annual plans at Month 3 for monthly subscribers.

### 📌 Insight 4: Referral & Email Channels Yield Best ROI
- Referral and Email acquisition channels produce customers with higher LTV and lower churn
- **Recommendation:** Increase investment in referral reward programmes and personalised email campaigns. Reduce spend on low-ROI channels.

### 📌 Insight 5: Support Tickets are a Churn Warning Signal
- Customers with 4+ support tickets show elevated churn probability
- **Recommendation:** Set up automated alerts when a customer raises a 3rd ticket within 30 days. Assign dedicated account managers for high-ticket customers.

### 📌 Insight 6: AI Churn Model Enables Early Intervention
- The Random Forest model predicts churn with **high accuracy and ROC-AUC**
- **Recommendation:** Deploy the model in production for weekly churn risk scoring. Export the top 500 at-risk customers weekly for the retention team to action.

---

## ✅ Project Conclusion

This project demonstrated a complete end-to-end data analytics pipeline:
- **Data Cleaning** ensured reliable analysis on 15,000+ customer records
- **EDA** revealed key distributions, correlations, and outliers
- **Segmentation Analysis** identified high-value customer profiles
- **Marketing Analysis** revealed channel-level ROI differences
- **ML Churn Model** (Random Forest) delivered actionable churn predictions

All findings are documented in `PrakashMaitiProjectReport.docx`.

---
*Project completed by **Prakash Maiti** — IBM SkillsBuild Data Analytics with AI Internship*